In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "0" 
os.environ["GRB_LICENSE_FILE"] = "/usr0/home/naveenr/gurobi.lic" 
os.environ['MKL_THREADING_LAYER'] = "GNU"
import torch 

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
from concept_abstraction.training import *
from concept_abstraction.selection import *
from concept_abstraction.concept_bank import *
from concept_abstraction.env_utils import *
from concept_abstraction.environments import *
from concept_abstraction.utils import *
from concept_abstraction.environments import ConceptEnv
import sys 
import argparse
import secrets
import numpy as np 
import random 
import os
from stable_baselines3 import PPO
import pickle
import resource
import time 

In [4]:
is_jupyter = 'ipykernel' in sys.modules
is_main = __name__ == "__main__"

In [5]:
if is_main:
    seed = 43
    environment_string = "cart_pole"
    gold_timesteps = 4_000_000
    training_timesteps = 250_000 
    num_concepts_selected = 12
    out_folder = "basic"
    method = "random" 


In [6]:
if is_main:
    concept_list, processed_concepts = get_concepts(environment_string,"human_selected_binary",seed)
    num_concepts_selected = min(num_concepts_selected,len(concept_list))
    ground_truth_env, ground_truth_gym_env = get_environment(environment_string, None, seed,processed_concepts=processed_concepts)   
    model_name = "../../results/models/env={}_training={}_seed={}.zip".format(environment_string,gold_timesteps,seed)
    if os.path.exists(model_name):
        groundtruth_model = PPO.load(model_name)
    two_stage_env, two_stage_gym_env = get_environment(environment_string,concept_list,seed,concept_idx=list(range(len(concept_list))),processed_concepts=processed_concepts)
    two_stage_env.reset().shape

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

In [9]:
rollout_q_estimates_td(groundtruth_model, ground_truth_gym_env, concept_list, states=None, gamma=0.99, 
                                total_timesteps=1_000, epsilon=0.1,
                                learning_rate=1e-4, update_freq=20, initial_random=0.3,
                                mimic=False, final_training=1_000, get_td_learner=False,use_initial_random=True)

[-0.03874473 -0.03200638  0.02986703  0.00417658]
[tensor(0.), tensor(0.), tensor(1.), tensor(1.), tensor(0.), tensor(0.), tensor(1.), tensor(1.), tensor(1.), tensor(1.), tensor(0.), tensor(0.)]
[-0.03146496 -0.02092309 -0.04798922 -0.03023816]
[tensor(0.), tensor(0.), tensor(1.), tensor(1.), tensor(0.), tensor(0.), tensor(0.), tensor(0.), tensor(1.), tensor(1.), tensor(0.), tensor(0.)]
[-0.00526681  0.04547949  0.01257507  0.02326351]
[tensor(1.), tensor(0.), tensor(1.), tensor(1.), tensor(0.), tensor(0.), tensor(1.), tensor(0.), tensor(1.), tensor(1.), tensor(0.), tensor(0.)]
[-0.01151331  0.02844539  0.01274199 -0.03575106]
[tensor(1.), tensor(0.), tensor(1.), tensor(1.), tensor(0.), tensor(0.), tensor(1.), tensor(0.), tensor(1.), tensor(1.), tensor(0.), tensor(0.)]
Starting stable training for sparse rewards...
Total of 1000 steps
Step 0/1000, Loss mean: 0.0000


KeyboardInterrupt: 

In [21]:
model_name = "../../results/q_estimates/env={}_training={}_seed={}_selection={}_source={}.pkl".format(environment_string,gold_timesteps,seed,"q_value","human_selected_binary")
if os.path.exists(model_name):
    q_estimates = pickle.load(open(model_name,"rb"))
else:
    q_estimates = rollout_q_estimates_td(groundtruth_model,ground_truth_gym_env,concept_list)
    pickle.dump(q_estimates,open(model_name,"wb"))


In [22]:
model_name = "../../results/models/concept_predictor_env={}_training={}_seed={}.pth".format(environment_string,100,seed)

height = width = 84

if environment_string == "mini_grid":
    num_frames = 1
else:
    num_frames = 4

if environment_string == "cart_pole":
    height = 160
    width = 240

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if os.path.exists(model_name):
    concept_predictor = ConceptPredictorCNN(len(concept_list), num_frames=num_frames,height=height,width=width).to(device)
    concept_predictor.load_state_dict(torch.load(model_name, weights_only=True))
    concept_predictor.eval()

In [25]:
two_stage_env, two_stage_gym_env = get_environment(environment_string,concept_list,seed,concept_idx=list(range(len(concept_list))),processed_concepts=processed_concepts)
model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps,custom_name="{}_imperfect_concepts".format(environment_string))

approx_kl,██▆▄▄▃▃▃▃▃▁▃▂▂▄▃▂▃▂▂▃▃▅▄▄▃▂▃▃▅▁▅▁▃▁▂▂▂▂▂
clip_fraction,█▅▃▂▂▂▂▂▂▂▄▂▂▁▁▁▁▃▃▄▅▅▂▇▅▃▂▃▂▃▄▄▃▂▂▁▂▃▃▃
ema_norm_reward,▁▁▁▁▁▂▁▁▂▂▃▅▅▇█▇██▇▆▆▄▅▄▅▅▅▅▆▅▅▅▅▆▆▆▆▆▆▇
entropy_loss,▁▄▄▄▅▇▇▇▇▇██▇▆▆▆▇▇▇▇▇▇▇▇▆▇▇▇▇▇▇██████▇▇▇
episode_length_mean,▁▁▁▁▁▁▂▃▃▃▄█▅▆▇██▁█▅▄▇▃▃▃▄▃▄▄▄▅▃▄▄▄▄▆▅▄▄
episode_reward_max,▁▁▁▁▁▂▁▂▂▂▂▂▁▃▂▃▄▅█▇▇▆▇▄▇▄▆▅▇▃▃▃▃▃▅▄█▄▄▃
episode_reward_mean,▁▁▁▁▁▁▁▂▂▁▃▃▄▆▂▄▃█▅▆▄▇▅▅▄▄▃▅▅▃▅█▅▅▅▄▆█▅▆
episode_reward_min,▁▁▁▁▁▁▁▁▁▂▁▄█▂▄▇▃▇█▄█▇▆█▅▅▄▄▄▃▃▃▄▃▇▄▄▅▃▅
episodes_completed,▁▁▁▁▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇████
explained_variance,▅▅▇▇▇▄▆▃▃▂▇▇▆▆▁▁▁▂▂▁▁▁▁▁▁▁▁▁▁▂▂██▆▄▄▃▃▁▃
+1,...


In [26]:
two_stage_env, two_stage_gym_env = get_environment(environment_string,concept_list,seed,fast_predictor=concept_predictor,use_processed=True,concept_idx=list(range(len(concept_list))),processed_concepts=processed_concepts)
model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps,custom_name="{}_imperfect_concepts".format(environment_string))

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

approx_kl,█▁▂▂▂▁▄▄▃▂▄▃▄▃▃▃▄▂▃▃▃▃▂▂▂▂▂▂▃▄▁▃▃▆▆▁▂▁▃▂
clip_fraction,▁▁▂▃▁▂▇▃▃▃▃▄▄▂▃▃▆▆▅▅▃▅▅██▃▆▁▂▅▁▄▅▆▆▁▁▁▁▂
ema_norm_reward,▁▁▁▁▁▂▂▂▃▃▃▃▃▃▃▃▄▄▅▅▃▄▆▇▆▆▆▇▆▆▆▇▆▇▇▇███▇
entropy_loss,▁▄▄▄▅▅▆▆▆▆▆▇██▇▇▇▇███▇█████▇▇▇▇▇▇▇▇▇██▇█
episode_length_mean,▁▁▁▁▁▁▂▂▂▃▃▃▂▃▃▂▃▂▄▅▃▆█▅▅▅██▃██████▄████
episode_reward_max,▁▁▁▁▁▂▁▂▃▂▃▂▃▃▂▃▃▃▃▃▆▃█▃▆█▄▄████▄███████
episode_reward_mean,▁▁▁▂▁▂▃▂▃▂▃▃▂▂▅▃▄▃▅▆█▃▃▃▇▅███▄▃▇████▅█▆█
episode_reward_min,▁▁▁▂▁▁▁▂▂▃▄▂▄▇▃▄▃▆▇███▆▃▄██▃▇█▃▄█████▆█▃
episodes_completed,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇████
explained_variance,▇▇▅▅▇▇▇▇▇▇▇█▇▇▇▇▇▇█▅▅▅███▇▇█▇▇▇▅▄▄▇▇▁▇▇█
+1,...


In [199]:
two_stage_env, two_stage_gym_env = get_environment(environment_string,concept_list,seed,processed_concepts=processed_concepts,concept_idx=list(range(len(concept_list))))
model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps,custom_name="{}_perfect_concepts".format(environment_string))    


approx_kl,▁▁▂▂▄▂▂▁▂▂▂▄▄▄▃▃▃▄▄▄▄▂▄▇▇▃▃▄▄▄▄▄▅▅█▆▆▆██
clip_fraction,▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▁▂▂▂▄▂███▄▃▅▅▅▃▃▅▅▅▅█
ema_norm_reward,▂▁▁▁▁▁▁▁▁▂▁▂▂▂▂▃▂▂▄▃▃▄▄▅▅▆▆▅▅▇▆▆▇▇▇▇████
entropy_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▃▃▄▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▇███
episode_length_mean,███████▃████▅█▅█▆▂███▃▂▂▃▂▃▅▃▃▃▂▂▅▂▂▂▂▂▁
episode_reward_max,▁▁▁▃▁▁▁▄▁▁▁█▁▃▁▅▅▁▁▆▁▄▄▁▅▅▇▁▃▄█████████▅
episode_reward_mean,▁▁▅▁▁▃▁▃▁▁▁▁▁▁▁▅▅▆▁▄▇▅▃▁▆▂▆▇█▇▇██▇██▇▇█▇
episode_reward_min,▆▁▃▁▁▁▁▁▆▃▁▁▁▄▁▁▇▅▅▁▁▁▁▅▇█▇▆▅██▇▇█▆▇█▇██
episodes_completed,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇████
explained_variance,▁▁▅▅▅▄▅▅▅▅▅▅▅▅▄▆▇▆▇▇▇▇▇▇▇▇▇▇▇███████████
+1,...


In [198]:
train_ppo_model(vec_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps,custom_name="{}_perfect_random".format(environment_string))        

approx_kl,▂▁▁▁▃▃▃▃▁▁▄▃▃▃▃▄▄▄▄█▅▃▃▅▅▅▅▅▆▅▄▄▅▅▅▆▆▆▆▆
clip_fraction,▁▁▁▁▁▁▁▁▁▁▁▃▁▁▃▃▄▄▄▄▁▁▂▂▃▃▃▄▄▄▃▄▃▅▅▅▅██▅
ema_norm_reward,▁▁▁▁▁▁▁▂▁▂▂▃▂▂▂▃▂▃▃▃▃▄▃▅▄▆▅▅▆▅▇▇▇▇▇█████
entropy_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▃▂▃▃▄▄▄▄▄▅▅▆▆▇▇▇▇██
episode_length_mean,████████▇██▄█▃▄▇▅▂▁▆▅▁▁▂▂█▃▃▁▃▂▂▁▂▂▁▁▁▂▁
episode_reward_max,▁▁▁▁▇▁▄▁▇▁▅▄▁▆▅█▁▆▇▆▆██▇█▇▁█████████████
episode_reward_mean,▁▁▁▁▁▁▁▁▁▄▁▄▄▄▅▂▂▆█▆▇▅██▆▇▇▆▇▇█▆█▆██▇▇█▇
episode_reward_min,▁▃▁▁▁▃▁▁▁▁▆▂▁▆▄▃▇▅▆▄▆▁▅▁▅▇▁▇▆▇▇▇▇▇██████
episodes_completed,▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▆▆▆▆▇▇▇▇▇█
explained_variance,▁▆▆▆▆▆▆▄▄▄▇▇▇▇▇▇▇████▇▇▇▇███████████████
+1,...


In [188]:
obs = [[1, 2, 2, 1, 3, 2, 2, 0, 0, 0, 0, 1], [1, 3, 3, 0, 0, 2, 1, 0, 0, 0, 0, 1], [1, 3, 2, 0, 0, 2, 1, 0, 0, 0, 0, 1], [1, 1, 2, 1, 3, 2, 1, 0, 0, 1, 0, 0], [1, 1, 3, 0, 0, 2, 1, 0, 0, 1, 0, 0], [1, 3, 0, 1, 2, 2, 1, 0, 0, 0, 0, 0], [1, 3, 3, 1, 2, 2, 1, 0, 0, 0, 0, 0], [1, 2, 3, 1, 3, 2, 1, 0, 0, 0, 0, 1]]
true = np.array([[c(o) for c in concept_list_old] for o in obs])
true.dtype


dtype('int64')